# 00 — Crawl Data từ Nhà Tốt

## Mục tiêu
- Crawl dữ liệu bất động sản từ nhatot.com cho các quận tại TP.HCM
- Lưu data thành file CSV riêng cho từng quận
- Chuẩn bị data cho phân tích và modeling

In [15]:
# Import thư viện cần thiết
from curl_cffi import requests as cffi_requests  # ← Giả lập TLS fingerprint Chrome
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
import re
import os
from urllib.parse import urljoin, urlencode
import warnings
warnings.filterwarnings('ignore')

print("✅ curl_cffi loaded — sẽ giả lập TLS fingerprint Chrome để bypass anti-bot")

✅ curl_cffi loaded — sẽ giả lập TLS fingerprint Chrome để bypass anti-bot


## 1. Setup và Configuration

In [16]:

# ─── Session dùng curl_cffi (giả lập Chrome TLS fingerprint) ───
# impersonate="chrome131" → TLS handshake giống hệt Chrome 131 thật
# Đây là lý do chính giúp bypass anti-bot (nhatot detect TLS fingerprint của requests)

session = cffi_requests.Session(impersonate="chrome131")

def warmup_session():
    """Truy cập trang chủ trước để lấy cookies — giống user thật"""
    try:
        print("🔄 Warming up session (Chrome TLS impersonation)...")
        resp = session.get('https://www.nhatot.com/', timeout=15)
        resp.raise_for_status()
        cookies = dict(resp.cookies)
        print(f"✅ Session ready! Status: {resp.status_code}, Cookies: {list(cookies.keys())[:5]}")
        return True
    except Exception as e:
        print(f"⚠️ Warmup failed: {e}")
        return False

# ─── Danh sách quận với slug cho URL ───
DISTRICTS = {
    96:  {'name': 'Quận 1',              'slug': 'quan-1'},
    98:  {'name': 'Quận 3',              'slug': 'quan-3'},
    99:  {'name': 'Quận 4',              'slug': 'quan-4'},
    100: {'name': 'Quận 5',              'slug': 'quan-5'},
    101: {'name': 'Quận 6',              'slug': 'quan-6'},
    102: {'name': 'Quận 7',              'slug': 'quan-7'},
    103: {'name': 'Quận 8',              'slug': 'quan-8'},
    105: {'name': 'Quận 10',             'slug': 'quan-10'},
    106: {'name': 'Quận 11',             'slug': 'quan-11'},
    107: {'name': 'Quận 12',             'slug': 'quan-12'},
    108: {'name': 'Quận Bình Tân',       'slug': 'quan-binh-tan'},
    109: {'name': 'Quận Bình Thạnh',     'slug': 'quan-binh-thanh'},
    110: {'name': 'Quận Gò Vấp',         'slug': 'quan-go-vap'},
    111: {'name': 'Quận Phú Nhuận',      'slug': 'quan-phu-nhuan'},
    112: {'name': 'Quận Tân Bình',       'slug': 'quan-tan-binh'},
    113: {'name': 'Quận Tân Phú',        'slug': 'quan-tan-phu'},
    114: {'name': 'Huyện Bình Chánh',    'slug': 'huyen-binh-chanh'},
    115: {'name': 'Huyện Cần Giờ',       'slug': 'huyen-can-gio'},
    116: {'name': 'Huyện Củ Chi',        'slug': 'huyen-cu-chi'},
    117: {'name': 'Huyện Hóc Môn',       'slug': 'huyen-hoc-mon'},
    118: {'name': 'Huyện Nhà Bè',        'slug': 'huyen-nha-be'},
    119: {'name': 'Thành phố Thủ Đức',   'slug': 'thanh-pho-thu-duc'},
}

# ─── Cấu hình crawling ───
MAX_PAGES_PER_DISTRICT = 300    # Tối đa 300 trang/quận (~20 ads/trang ≈ 6000 ads/quận)
DELAY_BETWEEN_PAGES = (2, 4)    # Delay giữa các trang (giây)
DELAY_BETWEEN_DISTRICTS = 5     # Delay giữa các quận (giây)
MAX_RETRIES = 3                 # Số lần retry khi bị lỗi
COOLDOWN_ON_BLOCK = 30          # Chờ 30s khi bị 403/429

# Tạo thư mục lưu data
os.makedirs('../data/raw', exist_ok=True)

print(f"📋 Cấu hình crawl:")
print(f"   • {len(DISTRICTS)} quận/thành phố")
print(f"   • Tối đa {MAX_PAGES_PER_DISTRICT} trang/quận (~20 ads/trang)")
print(f"   • Delay giữa trang: {DELAY_BETWEEN_PAGES[0]}-{DELAY_BETWEEN_PAGES[1]}s")
print(f"   • Delay giữa quận: {DELAY_BETWEEN_DISTRICTS}s")
print(f"   • TLS impersonation: chrome131\n")

warmup_session()

print()
for code, info in DISTRICTS.items():
    url = f"https://www.nhatot.com/mua-ban-bat-dong-san-{info['slug']}-tp-ho-chi-minh?page=1"
    print(f"  - {info['name']:25s} → {url}")


📋 Cấu hình crawl:
   • 22 quận/thành phố
   • Tối đa 300 trang/quận (~20 ads/trang)
   • Delay giữa trang: 2-4s
   • Delay giữa quận: 5s
   • TLS impersonation: chrome131

🔄 Warming up session (Chrome TLS impersonation)...
✅ Session ready! Status: 200, Cookies: ['_cfuvid']

  - Quận 1                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
  - Quận 3                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-3-tp-ho-chi-minh?page=1
  - Quận 4                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-4-tp-ho-chi-minh?page=1
  - Quận 5                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-5-tp-ho-chi-minh?page=1
  - Quận 6                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-6-tp-ho-chi-minh?page=1
  - Quận 7                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-7-tp-ho-chi-minh?page=1
  - Quận 8                    → https://www.nhatot.com/mua-ban-bat-dong-san-quan-8-tp-ho-chi-mi

## 2. Functions để Parse Data

In [17]:

def extract_price(price_text):
    """Extract và convert giá từ text thành số (đơn vị: tỷ)"""
    if not price_text:
        return None
        
    price_text = re.sub(r'[^\d.,\s]', '', str(price_text).strip())
    
    try:
        numbers = re.findall(r'[\d,]+(?:\.[\d]+)?', price_text)
        if not numbers:
            return None
            
        price_str = numbers[0].replace(',', '')
        price = float(price_str)
        
        if price > 1000:
            price = price / 1000
            
        return round(price, 2)
    except:
        return None

def extract_area(area_text):
    """Extract diện tích từ text thành số (đơn vị: m²)"""
    if not area_text:
        return None
        
    try:
        numbers = re.findall(r'[\d,]+(?:\.[\d]+)?', str(area_text))
        if not numbers:
            return None
            
        area_str = numbers[0].replace(',', '')
        return float(area_str)
    except:
        return None

def extract_rooms(room_text):
    """Extract số phòng ngủ từ text"""
    if not room_text:
        return None
        
    try:
        match = re.search(r'(\d+)\s*(?:phòng|PN)', str(room_text), re.IGNORECASE)
        if match:
            return int(match.group(1))
            
        numbers = re.findall(r'\d+', str(room_text))
        if numbers:
            return int(numbers[0])
            
        return None
    except:
        return None

def clean_text(text):
    """Clean và chuẩn hóa text — xử lý cả non-string"""
    if text is None:
        return ""
    text = str(text).strip()
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text)


## 3. Main Crawl Functions

In [18]:

BASE_WEB_URL = "https://www.nhatot.com/mua-ban-bat-dong-san"
API_BASE = "https://gateway.chotot.com/v1/public"


def build_page_url(slug, page=1):
    """Tạo URL trang listing theo quận và số trang
    Ví dụ: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
    """
    return f"{BASE_WEB_URL}-{slug}-tp-ho-chi-minh?page={page}"


def retry_with_backoff(func, *args, max_retries=MAX_RETRIES):
    """Retry với exponential backoff khi bị 403/429"""
    result = None
    for attempt in range(max_retries):
        result = func(*args)
        if result[2] not in (403, 429):
            return result
        wait = COOLDOWN_ON_BLOCK * (attempt + 1)
        print(f"    ⏸️  Bị block (HTTP {result[2]})! Chờ {wait}s... (lần {attempt+1}/{max_retries})")
        time.sleep(wait)
        warmup_session()
    return result


# ═══════════════════════════════════════════════
# Cách 1: Web Scraping — parse __NEXT_DATA__
# curl_cffi tự xử lý TLS + headers giống Chrome
# ═══════════════════════════════════════════════

def get_page_ads_web(slug, page=1):
    """Lấy ads bằng web scraping: fetch HTML → parse __NEXT_DATA__ (Next.js)
    Returns: (list_of_ads, has_more, error_code)
    """
    url = build_page_url(slug, page)
    try:
        resp = session.get(url, timeout=15)
        if resp.status_code in (403, 429):
            return [], False, resp.status_code
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, 'html.parser')
        script = soup.find('script', id='__NEXT_DATA__')
        if not script or not script.string:
            return [], False, None

        next_data = json.loads(script.string)
        ads = _extract_ads_from_next_data(next_data)
        return ads, len(ads) > 0, None

    except Exception as e:
        err_str = str(e)
        if '403' in err_str or '429' in err_str:
            code = 403 if '403' in err_str else 429
            return [], False, code
        if '404' in err_str:
            return [], False, None
        print(f"    ❌ Web error (page {page}): {e}")
        return [], False, None


def _extract_ads_from_next_data(next_data):
    """Trích ads từ __NEXT_DATA__ JSON (thử nhiều path)"""
    try:
        pp = next_data.get('props', {}).get('pageProps', {})
        for path_fn in [
            lambda p: p.get('initialState', {}).get('adListing', {}).get('ads', []),
            lambda p: p.get('adListing', {}).get('ads', []),
            lambda p: p.get('ads', []) if isinstance(p.get('ads'), list) else [],
            lambda p: p.get('initialProps', {}).get('adListing', {}).get('ads', []),
            lambda p: p.get('data', {}).get('ads', []) if isinstance(p.get('data', {}).get('ads'), list) else [],
        ]:
            try:
                ads = path_fn(pp)
                if ads:
                    return ads
            except:
                continue
        return _deep_find_ads(next_data)
    except:
        return []


def _deep_find_ads(obj, depth=0):
    """Tìm đệ quy key 'ads' chứa list > 3 items"""
    if depth > 6:
        return []
    if isinstance(obj, dict):
        if 'ads' in obj and isinstance(obj['ads'], list) and len(obj['ads']) > 3:
            return obj['ads']
        for val in obj.values():
            result = _deep_find_ads(val, depth + 1)
            if result:
                return result
    return []


# ═══════════════════════════════════════════════
# Cách 2: Chotot API (fallback)
# ═══════════════════════════════════════════════

def get_page_ads_api(area_code, page=1, limit=20):
    """Lấy ads từ Chotot API — fallback khi web scrape thất bại
    Returns: (list_of_ads, has_more, error_code)
    """
    url = f"{API_BASE}/ad-listing"
    offset = (page - 1) * limit
    params = {
        'cg': 1000, 'region': 13, 'area': area_code,
        'o': offset, 'limit': limit, 'st': 's,k', 'w': 1,
    }
    try:
        resp = session.get(url, params=params, timeout=10)
        if resp.status_code in (403, 429):
            return [], False, resp.status_code
        resp.raise_for_status()
        ads = resp.json().get('ads', [])
        return ads, len(ads) > 0, None
    except Exception as e:
        err_str = str(e)
        if '403' in err_str or '429' in err_str:
            code = 403 if '403' in err_str else 429
            return [], False, code
        print(f"    ❌ API error (page {page}): {e}")
        return [], False, None


# ═══════════════════════════════════════════════
# Hàm tổng hợp: web scrape → API fallback
# ═══════════════════════════════════════════════

def get_page_ads(slug, area_code, page=1):
    """Lấy ads từ 1 trang — ưu tiên web, fallback API
    Returns: (list_of_ads, has_more, method_used)
    """
    # Thử web scrape trước
    ads, has_more, err = retry_with_backoff(get_page_ads_web, slug, page)
    if ads:
        return ads, has_more, 'web'

    # Fallback → API
    if err in (403, 429):
        print(f"    🔄 Web bị block, chuyển sang API...")

    ads, has_more, err = retry_with_backoff(get_page_ads_api, area_code, page)
    if ads:
        return ads, has_more, 'api'

    if err in (403, 429):
        print(f"    🚫 Cả web và API đều bị block!")

    return [], False, None


# ─── Kiểm tra kết nối ───
print("🔌 Kiểm tra kết nối (curl_cffi + Chrome TLS)...")
test_info = DISTRICTS[96]
test_url = build_page_url(test_info['slug'], 1)
print(f"   URL: {test_url}")

test_ads, has_more, method = get_page_ads(test_info['slug'], 96, page=1)
if test_ads:
    print(f"✅ Thành công! Lấy được {len(test_ads)} ads từ page 1 (method: {method})")
    print(f"   Sample keys: {list(test_ads[0].keys())[:10]}")
else:
    print("❌ Không kết nối được!")


🔌 Kiểm tra kết nối (curl_cffi + Chrome TLS)...
   URL: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
✅ Thành công! Lấy được 20 ads từ page 1 (method: web)
   Sample keys: ['account_id', 'account_name', 'account_oid', 'ad_features', 'ad_id', 'ad_labels', 'area', 'area_name', 'area_v2', 'avatar']


In [19]:

def parse_ad_to_row(ad, district_name):
    """Parse JSON từ API thành row chuẩn giống file CSV"""
    # Giá (convert từ VNĐ → tỷ)
    price = ad.get('price', None)
    price_ty = round(price / 1e9, 3) if isinstance(price, (int, float)) and price > 0 else None

    # Diện tích (field 'size' hoặc 'area' — dùng 'size' cho m²)
    dien_tich = ad.get('size', ad.get('area', None))

    # Số phòng ngủ — có thể lấy trực tiếp từ field 'rooms'
    rooms = ad.get('rooms', None)
    if rooms is not None:
        try:
            rooms = int(rooms)
        except:
            rooms = None

    # Các thông số trong 'params' list (loại hình, giấy tờ)
    loai_hinh, giay_to = '', ''
    for p in ad.get('params', []):
        pid = p.get('id', '')
        val = p.get('value', '')
        label = p.get('label', '')
        if pid == 'property_type':
            loai_hinh = clean_text(val or label)
        elif pid == 'property_legal_document':
            giay_to = clean_text(val or label)
        # rooms backup từ params nếu chưa có
        elif pid == 'rooms' and rooms is None:
            try:
                rooms = int(val)
            except:
                pass

    # Giấy tờ cũng có thể ở field trực tiếp
    if not giay_to:
        giay_to = clean_text(ad.get('property_legal_document', ''))

    # Loại hình từ house_type
    if not loai_hinh:
        loai_hinh = clean_text(ad.get('house_type', ''))

    ad_id = ad.get('list_id', '')
    return {
        'url':             f"https://www.nhatot.com/{ad_id}",
        'tieu_de':         clean_text(ad.get('subject', '')),
        'gia_ban':         price_ty,
        'dien_tich':       dien_tich,
        'so_phong_ngu':    rooms,
        'loai_hinh':       loai_hinh,
        'dia_chi':         clean_text(ad.get('ward_name', ad.get('area_name', ''))),
        'mo_ta':           clean_text(ad.get('body', '')),
        'giay_to_phap_ly': giay_to,
        'quan':            district_name,
    }


## 4. Main Crawling Process

In [20]:

def crawl_district_data(area_code, district_info, max_pages=MAX_PAGES_PER_DISTRICT):
    """Crawl data một quận — duyệt theo từng page URL

    URL: https://www.nhatot.com/mua-ban-bat-dong-san-{slug}-tp-ho-chi-minh?page={N}
    Ưu tiên web scrape, fallback API. Dừng khi hết data (3 trang trống liên tiếp).
    """
    slug = district_info['slug']
    name = district_info['name']

    print(f"\n🏘️  Crawling {name} (area_code={area_code})")
    print(f"   URL: {build_page_url(slug, 1)}")

    all_rows = []
    seen_urls = set()
    method_stats = {'web': 0, 'api': 0}
    consecutive_empty = 0

    for page in range(1, max_pages + 1):
        print(f"  📄 Page {page}: {build_page_url(slug, page)}")

        ads, has_more, method = get_page_ads(slug, area_code, page)

        if not ads:
            consecutive_empty += 1
            print(f"       ⚠️ Không có data (trống {consecutive_empty}/3)")
            if consecutive_empty >= 3:
                print(f"  ✅ Hết data (3 trang trống liên tiếp)")
                break
            continue

        consecutive_empty = 0
        if method:
            method_stats[method] = method_stats.get(method, 0) + 1

        # Parse ads → rows, loại trùng theo URL
        page_new = 0
        for ad in ads:
            row = parse_ad_to_row(ad, name)
            if row['url'] not in seen_urls:
                seen_urls.add(row['url'])
                all_rows.append(row)
                page_new += 1

        print(f"       ✅ +{page_new} mới (total: {len(all_rows)}) [{method}]")

        if not has_more:
            print(f"  ✅ Trang cuối cùng")
            break

        # Delay giữa các trang
        delay = random.uniform(*DELAY_BETWEEN_PAGES)
        if page % 10 == 0:
            extra = random.uniform(5, 10)
            print(f"       ⏳ Nghỉ thêm {extra:.0f}s (mỗi 10 trang)...")
            delay += extra
        time.sleep(delay)

    if all_rows:
        df = pd.DataFrame(all_rows)
        print(f"\n✅ Xong {name}: {len(df)} records (web={method_stats.get('web',0)}, api={method_stats.get('api',0)} pages)")
        return df
    else:
        print(f"\n❌ Không lấy được data cho {name}")
        return pd.DataFrame()


def test_crawl_sample():
    """Test nhanh với Quận 1 — crawl 3 trang đầu"""
    print("🧪 Testing crawl Quận 1 (3 trang đầu)...")
    print(f"   Page 1: {build_page_url('quan-1', 1)}")
    print(f"   Page 2: {build_page_url('quan-1', 2)}")
    print(f"   Page 3: {build_page_url('quan-1', 3)}\n")

    warmup_session()
    df = crawl_district_data(96, DISTRICTS[96], max_pages=3)

    if not df.empty:
        print(f"\n📊 Sample ({len(df)} records):")
        cols = [c for c in ['tieu_de', 'gia_ban', 'dien_tich', 'so_phong_ngu', 'quan'] if c in df.columns]
        print(df[cols].head(10).to_string())
    else:
        print("❌ Không lấy được data. Thử đổi mạng/VPN hoặc chờ 30 phút.")

    return df


In [21]:
# Chạy test trước khi crawl full
test_df = test_crawl_sample()

🧪 Testing crawl Quận 1 (3 trang đầu)...
   Page 1: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
   Page 2: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=2
   Page 3: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=3

🔄 Warming up session (Chrome TLS impersonation)...
✅ Session ready! Status: 200, Cookies: []

🏘️  Crawling Quận 1 (area_code=96)
   URL: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
  📄 Page 1: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
       ✅ +20 mới (total: 20) [web]
  📄 Page 2: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=2
       ✅ +20 mới (total: 40) [web]
  📄 Page 3: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=3
       ✅ +20 mới (total: 60) [web]

✅ Xong Quận 1: 60 records (web=3, api=0 pages)

📊 Sample (60 records):
                                                                 

## 5. Crawl All Districts

In [22]:

# Crawl tất cả các quận (chỉ chạy khi test thành công)
if 'test_df' in locals() and not test_df.empty:
    print("✅ Test crawl thành công! Bắt đầu crawl full...")
    print(f"📋 Tối đa {MAX_PAGES_PER_DISTRICT} trang/quận × {len(DISTRICTS)} quận\n")

    crawled_districts = {}
    failed_districts = []

    for i, (area_code, district_info) in enumerate(DISTRICTS.items()):
        try:
            print(f"\n{'='*60}")
            print(f"🏘️  [{i+1}/{len(DISTRICTS)}] {district_info['name']} (area_code={area_code})")
            print(f"   URL: {build_page_url(district_info['slug'], 1)}")
            print(f"{'='*60}")

            df = crawl_district_data(area_code, district_info, max_pages=MAX_PAGES_PER_DISTRICT)

            if not df.empty:
                filename = f"../data/raw/quan-{area_code}-{district_info['slug']}.csv"
                df.to_csv(filename, index=False, encoding='utf-8-sig')
                crawled_districts[district_info['name']] = {
                    'dataframe': df, 'count': len(df), 'filename': filename,
                }
                print(f"💾 Saved {len(df)} records → {filename}")
            else:
                failed_districts.append(district_info['name'])
                print(f"❌ No data for {district_info['name']}")

            # Delay giữa các quận
            wait = DELAY_BETWEEN_DISTRICTS + random.uniform(0, 5)
            print(f"⏳ Waiting {wait:.0f}s...")
            time.sleep(wait)

        except Exception as e:
            print(f"❌ Error crawling {district_info['name']}: {e}")
            failed_districts.append(district_info['name'])
            continue

    # Summary
    print(f"\n{'='*60}")
    print(f"📊 CRAWLING SUMMARY")
    print(f"{'='*60}")
    print(f"✅ Successful: {len(crawled_districts)} quận")
    print(f"❌ Failed:     {len(failed_districts)} quận")

    total = 0
    if crawled_districts:
        print(f"\n📈 Chi tiết:")
        for name, info in crawled_districts.items():
            print(f"  - {name:25s}: {info['count']:>4d} records → {info['filename']}")
            total += info['count']
        print(f"\n🎯 Tổng cộng: {total:,} records")

    if failed_districts:
        print(f"\n⚠️  Failed: {', '.join(failed_districts)}")
else:
    print("❌ Test chưa chạy hoặc thất bại. Vui lòng chạy test cell trước.")


✅ Test crawl thành công! Bắt đầu crawl full...
📋 Tối đa 300 trang/quận × 22 quận


🏘️  [1/22] Quận 1 (area_code=96)
   URL: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1

🏘️  Crawling Quận 1 (area_code=96)
   URL: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
  📄 Page 1: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=1
       ✅ +20 mới (total: 20) [web]
  📄 Page 2: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=2
       ✅ +20 mới (total: 40) [web]
  📄 Page 3: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=3
       ✅ +20 mới (total: 60) [web]
  📄 Page 4: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=4
       ✅ +20 mới (total: 80) [web]
  📄 Page 5: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=5
       ✅ +20 mới (total: 100) [web]
  📄 Page 6: https://www.nhatot.com/mua-ban-bat-dong-san-quan-1-tp-ho-chi-minh?page=6


## 6. Data Validation & Export

In [24]:
# Combine tất cả data và validate
if 'crawled_districts' in locals() and crawled_districts:
    print("🔍 Validating và combining crawled data...")
    
    # Combine tất cả DataFrames
    all_dfs = [info['dataframe'] for info in crawled_districts.values()]
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    print(f"📊 Combined dataset: {len(combined_df)} total records")
    print(f"📍 Districts: {combined_df['quan'].unique()}")
    
    # Data quality checks
    print(f"\n🔍 Data Quality Assessment:")
    
    # Missing values
    print(f"\n1️⃣ Missing values:")
    missing_stats = combined_df.isnull().sum()
    for col, missing in missing_stats.items():
        if missing > 0:
            pct = (missing / len(combined_df)) * 100
            print(f"  - {col}: {missing} ({pct:.1f}%)")
    
    # Price validation
    valid_prices = combined_df['gia_ban'].notna()
    print(f"\n2️⃣ Price data:")
    print(f"  - Valid prices: {valid_prices.sum()} ({valid_prices.mean()*100:.1f}%)")
    if valid_prices.sum() > 0:
        price_stats = combined_df[valid_prices]['gia_ban'].describe()
        print(f"  - Price range: {price_stats['min']:.1f} - {price_stats['max']:.1f} tỷ")
        print(f"  - Mean price: {price_stats['mean']:.1f} tỷ")
    
    # Area validation
    valid_areas = combined_df['dien_tich'].notna()
    print(f"\n3️⃣ Area data:")
    print(f"  - Valid areas: {valid_areas.sum()} ({valid_areas.mean()*100:.1f}%)")
    if valid_areas.sum() > 0:
        area_stats = combined_df[valid_areas]['dien_tich'].describe()
        print(f"  - Area range: {area_stats['min']:.0f} - {area_stats['max']:.0f} m²")
        print(f"  - Mean area: {area_stats['mean']:.0f} m²")
    
    # Records by district
    print(f"\n4️⃣ Records per district:")
    district_counts = combined_df['quan'].value_counts()
    for district, count in district_counts.items():
        print(f"  - {district}: {count} records")
    
    # Save combined dataset
    combined_filename = "../data/raw/all_districts_combined.csv"
    combined_df.to_csv(combined_filename, index=False, encoding='utf-8-sig')
    print(f"\n💾 Saved combined dataset: {combined_filename}")
    
    # Sample data preview
    print(f"\n📋 Sample data preview:")
    preview_cols = ['quan', 'tieu_de', 'gia_ban', 'dien_tich', 'so_phong_ngu']
    available_cols = [col for col in preview_cols if col in combined_df.columns]
    print(combined_df[available_cols].head(10))
    
else:
    print("❌ No crawled data available. Run the crawling process first.")

🔍 Validating và combining crawled data...
📊 Combined dataset: 59590 total records
📍 Districts: ['Quận 1' 'Quận 3' 'Quận 4' 'Quận 5' 'Quận 6' 'Quận 7' 'Quận 8' 'Quận 10'
 'Quận 11' 'Quận 12' 'Quận Bình Tân' 'Quận Bình Thạnh' 'Quận Gò Vấp'
 'Quận Phú Nhuận' 'Quận Tân Bình' 'Quận Tân Phú' 'Huyện Bình Chánh'
 'Huyện Cần Giờ' 'Huyện Củ Chi' 'Huyện Hóc Môn' 'Huyện Nhà Bè'
 'Thành phố Thủ Đức']

🔍 Data Quality Assessment:

1️⃣ Missing values:
  - gia_ban: 1472 (2.5%)
  - so_phong_ngu: 13848 (23.2%)

2️⃣ Price data:
  - Valid prices: 58118 (97.5%)
  - Price range: 0.0 - 1250.0 tỷ
  - Mean price: 7.3 tỷ

3️⃣ Area data:
  - Valid areas: 59590 (100.0%)
  - Area range: 1 - 88170 m²
  - Mean area: 124 m²

4️⃣ Records per district:
  - Thành phố Thủ Đức: 5993 records
  - Quận Bình Tân: 5987 records
  - Quận 12: 5280 records
  - Quận Gò Vấp: 4813 records
  - Quận Tân Phú: 3650 records
  - Quận Bình Thạnh: 3520 records
  - Quận Tân Bình: 3145 records
  - Quận 7: 3076 records
  - Quận 8: 2949 records
 

## 📝 Hướng dẫn sử dụng

### 🚀 **Cách chạy crawler:**

1. **Chạy test trước**: Execute cell "Test crawl" để kiểm tra system
2. **Chạy full crawl**: Nếu test OK, execute cell "Crawl all districts"  
3. **Validate data**: Execute cell cuối để kiểm tra chất lượng data

### 📊 **Output files:**

- **Riêng lẻ**: `../data/raw/quan-{ten-quan}.csv` cho từng quận
- **Combined**: `../data/raw/all_districts_combined.csv` (tất cả quận)

```

### 🛡️ **Anti-blocking features:**

- Random delays giữa requests (2-4 seconds)
- Realistic User-Agent headers
- Session management  
- Error handling & retry logic

### 📋 **Data schema:**

| Column | Type | Description |
|--------|------|-------------|
| `url` | str | Link tin đăng |
| `tieu_de` | str | Tiêu đề tin đăng |
| `gia_ban` | float | Giá bán (tỷ VNĐ) |
| `dien_tich` | float | Diện tích (m²) |
| `so_phong_ngu` | int | Số phòng ngủ |
| `loai_hinh` | str | Loại hình nhà |
| `dia_chi` | str | Địa chỉ |
| `mo_ta` | str | Mô tả chi tiết |
| `giay_to_phap_ly` | str | Loại giấy tờ |
| `quan` | str | Quận/Thành phố |

**⚠️ Lưu ý**: Crawler có thể mất 30-60 phút để hoàn thành tất cả quận. Không dừng giữa chừng để tránh mất data!